# ЛР 3 - можно ли доверять LLM в задачах классического машинного обучения

Модели: GigaChat-3-Ultra, GPT-6.1 Terra (API) и Qwen3-4B-Instruct-2507 (локально на гугл колабе)

# Гигачат

GigaChat-3-Ultra. Ключ: `GIGACHAT_CREDENTIALS` в `.env`.

In [19]:
import os
from dotenv import load_dotenv

load_dotenv(".env");

from gigachat import GigaChat

giga_client = GigaChat(
    base_url="https://api.giga.chat/v1",
    credentials=os.environ["GIGACHAT_CREDENTIALS"],
    scope=os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_PERS"),
    model="GigaChat-3-Ultra",
    timeout=60,
)

def gigachat(prompt):
    response = giga_client.chat({
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 2048,
    })
    return response.choices[0].message.content

In [20]:
prompt = "В двух предложениях объясни, что такое переобучение."
print(gigachat(prompt))

KeyboardInterrupt: 

# ChatGPT

GPT-6.1 Terra через LLM Gate. Ключ: `LLM_GATE_API_KEY` в `.env`.

Terra заменяет Sol из-за ошибок шлюза. Прежние ответы Sol сохранены в `answers_gpt_main.jsonl`; новый прогон — в `answers_terra_main.jsonl`.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env", override=True);

from openai import OpenAI

gpt_client = OpenAI(
    base_url="https://api.llm-gate.space/v1",
    api_key=os.environ["LLM_GATE_API_KEY"],
    timeout=60,
    max_retries=3,
)

def chatgpt(prompt):
    response = gpt_client.chat.completions.create(
        model="gpt-6.1-terra",
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort="low",
        max_completion_tokens=2048,
    )
    return response.choices[0].message.content

In [ ]:
prompt = "В двух предложениях объясни, что такое переобучение."
print(chatgpt(prompt))

# Общая функция и лог

`ask("gigachat" или "gpt", prompt, temperature)` возвращает ответ, общее число токенов и время в секундах. Ответы отладки сохраняются в `otladka/answers_giga_terra_otladka.json`. Полный прогон сохраняется отдельно для каждой модели.

In [38]:
import json
import time

LOG = "otladka/answers_giga_terra_otladka.json"

SYSTEM_PROMPT = """Реши предложенную задачу. При необходимости приведи объяснение или код.
Если данных недостаточно для ответа, прямо укажи это.
Последнюю строку оформи строго так:
ОТВЕТ: <краткий итог> | УВЕРЕННОСТЬ: <целое число от 0 до 100>"""

def ask(model, prompt, temperature):
    start = time.perf_counter()
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
    if model == "gigachat":
        response = giga_client.chat({
            "messages": messages, "temperature": temperature, "max_tokens": 2048})
        tokens = response.usage.total_tokens
    elif model == "gpt":
        response = gpt_client.chat.completions.create(
            model="gpt-6.1-terra", messages=messages, temperature=temperature,
            reasoning_effort="low", max_completion_tokens=2048)
        tokens = response.usage.total_tokens
    seconds = round(time.perf_counter() - start, 3)
    answer = response.choices[0].message.content
    record = {"model": model, "model_id": response.model, "prompt": prompt, "temperature": temperature,
              "answer": answer, "tokens": tokens, "seconds": seconds}
    with open(LOG, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
    return answer, tokens, seconds

In [27]:
answer, tokens, seconds = ask("gigachat", "Что такое переобучение? Ответь кратко.", 0.2)
print(answer)
print(f"Токены: {tokens}, время: {seconds} с")

Переобучение — это когда модель машинного обучения слишком хорошо подстраивается под обучающие данные, запоминая их шум и случайные флуктуации вместо выявления общих закономерностей. В результате модель показывает отличные результаты на данных, которые она уже видела, но плохо справляется с новыми, ранее неизвестными ей данными.

ОТВЕТ: Слишком сильная подгонка модели под обучающие данные, из-за которой ухудшается её способность к обобщению на новых данных. | УВЕРЕННОСТЬ: 100
Токены: 200, время: 1.171 с


# Отладка промпта

Пять задач из `test_bank.csv`, температура 0.

In [ ]:
import re
import csv 
model = "gpt"  # gigachat

with open("test_bank.csv", encoding="utf-8-sig", newline="") as file:
    for task in csv.DictReader(file):
        answer, tokens, seconds = ask(model, task["текст_ru"], 0)
        last_line = answer.strip().splitlines()[-1]
        ok = bool(re.fullmatch(r"ОТВЕТ: .+ \| УВЕРЕННОСТЬ: (?:100|[1-9]?[0-9])", last_line))
        print(task["id"], "формат OK" if ok else "формат нарушен")
        print(answer, "\n")

T01 формат OK
Для решения этой задачи необходимо определить, что представляет собой бинарное дерево с максимальным количеством листьев при заданной глубине.

## Определение терминов

*   **Бинарное дерево** — это структура данных, в которой каждый узел имеет не более двух потомков (левого и правого).
*   **Глубина узла** — это количество рёбер на пути от корня до этого узла. По условию, корень имеет глубину 0.
*   **Глубина дерева** — это максимальная глубина среди всех его узлов. В данном случае глубина дерева равна 4. Это означает, что самый длинный путь от корня до листа состоит из 4 рёбер, а самый глубокий лист находится на уровне 4.
*   **Лист** — это узел, у которого нет потомков.

## Логика решения

Чтобы количество листьев было максимальным, дерево должно быть **полным** (или совершенным). Это означает, что каждый уровень дерева, кроме последнего, должен быть полностью заполнен, а все узлы на последнем уровне должны быть листьями. Если на каком-либо уровне будет отсутствовать х

# Полный банк

Только `questions.csv`, температура 0. Каждый ответ сразу сохраняется. Повторный запуск пропускает сохранённые ответы. Не меняйте банк и системный промпт между запусками.


In [ ]:
from pathlib import Path

def run_bank(model):
    global LOG
    LOG = Path("answers_terra_main.jsonl" if model == "gpt" else "answers_gigachat_main.jsonl")
    done = {json.loads(line)["prompt"] for line in LOG.read_text(encoding="utf-8").splitlines()} if LOG.exists() else set()
    with open("questions.csv", encoding="utf-8-sig", newline="") as file:
        for task in csv.DictReader(file):
            if task["текст_ru"] in done:
                continue
            answer, tokens, seconds = ask(model, task["текст_ru"], 0)
            print(task["id"], model, "готово", tokens, "токенов,", seconds, "с")

#run_bank("gigachat")
run_bank("gpt")


A01 gigachat готово 656 токенов, 3.588 с
A02 gigachat готово 408 токенов, 1.996 с
A03 gigachat готово 981 токенов, 5.556 с
A04 gigachat готово 317 токенов, 2.23 с
A05 gigachat готово 574 токенов, 3.013 с
A06 gigachat готово 985 токенов, 5.09 с
B01 gigachat готово 377 токенов, 2.263 с
B02 gigachat готово 464 токенов, 3.19 с
B03 gigachat готово 312 токенов, 2.22 с
B04 gigachat готово 182 токенов, 1.367 с
B05 gigachat готово 324 токенов, 2.034 с
B06 gigachat готово 351 токенов, 2.597 с
C01 gigachat готово 937 токенов, 5.557 с
C02 gigachat готово 520 токенов, 3.529 с
C03 gigachat готово 1148 токенов, 8.05 с
C04 gigachat готово 538 токенов, 4.009 с
C05 gigachat готово 1097 токенов, 6.949 с
C06 gigachat готово 372 токенов, 2.319 с
D01 gigachat готово 444 токенов, 3.626 с
D02 gigachat готово 801 токенов, 4.864 с
D03 gigachat готово 730 токенов, 4.198 с
D04 gigachat готово 1216 токенов, 6.975 с
D05 gigachat готово 749 токенов, 4.599 с
D06 gigachat готово 904 токенов, 4.966 с
E01 gigachat готово 628 токенов, 4.699 с
E02 gigachat готово 812 токенов, 6.012 с
E03 gigachat готово 695 токенов, 4.995 с
F01 gigachat готово 912 токенов, 5.669 с
F02 gigachat готово 638 токенов, 3.786 с
F03 gigachat готово 538 токенов, 3.268 с
F04 gigachat готово 522 токенов, 2.777 с
F05 gigachat готово 360 токенов, 2.634 с
F06 gigachat готово 774 токенов, 4.849 с
F07 gigachat готово 668 токенов, 4.975 с
F08 gigachat готово 318 токенов, 2.118 с
F09 gigachat готово 616 токенов, 3.45 с



A01 qwen готово 388 токенов, 15.016 с
A02 qwen готово 426 токенов, 15.373 с
A03 qwen готово 616 токенов, 24.217 с
A04 qwen готово 470 токенов, 18.449 с
A05 qwen готово 842 токенов, 36.212 с
A06 qwen готово 1093 токенов, 50.746 с
B01 qwen готово 454 токенов, 17.045 с
B02 qwen готово 551 токенов, 22.837 с
B03 qwen готово 335 токенов, 11.029 с
B04 qwen готово 275 токенов, 7.959 с
B05 qwen готово 343 токенов, 12.121 с
B06 qwen готово 482 токенов, 18.777 с
C01 qwen готово 892 токенов, 38.951 с
C02 qwen готово 588 токенов, 24.1 с
C03 qwen готово 775 токенов, 32.931 с
C04 qwen готово 573 токенов, 22.706 с
C05 qwen готово 763 токенов, 32.165 с
C06 qwen готово 592 токенов, 23.95 с
D01 qwen готово 302 токенов, 9.175 с
D02 qwen готово 1129 токенов, 49.932 с
D03 qwen готово 1268 токенов, 59.23 с
D04 qwen готово 2209 токенов, 110.922 с
D05 qwen готово 1996 токенов, 98.206 с
D06 qwen готово 1985 токенов, 97.399 с
E01 qwen готово 433 токенов, 15.123 с
E02 qwen готово 413 токенов, 13.98 с
E03 qwen готово 507 токенов, 18.949 с
F01 qwen готово 789 токенов, 34.307 с
F02 qwen готово 712 токенов, 28.98 с
F03 qwen готово 616 токенов, 23.485 с
F04 qwen готово 878 токенов, 37.612 с
F05 qwen готово 419 токенов, 14.421 с
F06 qwen готово 750 токенов, 31.831 с
F07 qwen готово 2220 токенов, 134.094 с
F08 qwen готово 437 токенов, 15.744 с
F09 qwen готово 813 токенов, 34.034 с